# A3.7 · The agent gateway: one choke point when you scale

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.6 · Human approval that survives volume](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**.

| | |
|---|---|
| Tools used | agentgateway, OPA, Keycloak |

## What this lesson is

**What it covers.** Route every call through one gateway and show the same policy holding for agents that never implemented it.

**Why a security engineer needs it.** Per-agent controls diverge as the fleet grows, and legacy downstreams force a static credential back into agent code. The control it builds is: a single enforcement point holding identity, policy, egress, budget and audit — with the credential for legacy systems held there rather than by the agent.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

At one agent the controls live in the agent. At fifty, each team implements them slightly differently, none of them is audited, and the only honest answer to "is default-deny on?" is "in some of them".

> **At CyberTravels.** At four agents the controls live in the agents. When CyberTravels ships the eighth, nobody can answer “is default-deny on?” with anything better than “in some of them”. R9.

## 2 · The framework

```
   one agent                     fifty agents
   +--------------+              +-----+ +-----+ +-----+ +-----+
   | controls in  |              | a1  | | a2  | | ... | | a50 |
   | the agent    |              +--+--+ +--+--+ +--+--+ +--+--+
   +--------------+                 |       |       |       |
                                    +-------+---+---+-------+
                                                v
                                        +--------------+
                                        |   gateway    | identity, policy,
                                        +------+-------+ budget, egress, log
                                               v
                                            tools

   one place to enforce, one place to audit, one place to turn off
```

**Mitigates: every threat in this chapter, at one enforcement point.**

Everything in Chapters 2 and 3 works. The problem is where it lives.

At one agent, the controls sit in the agent, and that is fine. At fifty, it
stops being fine for reasons that have nothing to do with security engineering:

- Each team implements provenance, budgets and egress slightly differently.
- Nobody can answer "is this control on, everywhere" without reading fifty
  repositories.
- A new agent starts at zero and re-earns every control by hand.
- Fixing a control means fifty pull requests and a migration.

The **gateway** is the same controls, moved to a point every call must pass
through. It holds identity (A2.1–A2.3), policy (A3.1), egress (A3.3), budgets
(A3.4) and audit (A2.7). An agent that implements none of them still gets all of
them, because the enforcement is no longer the agent's responsibility.

It also solves a problem nothing else does: **downstream systems that cannot
consume a delegated identity.** A legacy database or a vendor API that only
understands a static credential forces that credential back into agent code —
undoing A2.3 completely. The gateway holds it instead, authorises the *user*
before the call, and presents the static credential onward. The agent never sees
it.

The honest cost: the gateway is now a single point of failure and a very
attractive target. It has to be operated accordingly.

> **What this control closes.**
>
> Not a new control. The same controls, at a point every call passes through — and the only answer to a downstream that cannot consume delegated identity.

## 3 · Proving every call goes through it, as a skill

A gateway is only a choke point if nothing routes around it, and "nothing routes around it" is not a fact you can read off a route table. The procedure tests reachability from inside the deployment, covers the paths people forget — tool-initiated calls, background jobs, retries — and records each guardrail's **action**, because a filter set to observe is a filter that is switched on and stopping nothing. This is the file in this repository:

### The skill — [`skills/attestation/llm-gateway-guardrail-verifier/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/llm-gateway-guardrail-verifier/SKILL.md)

```yaml
name: llm-gateway-guardrail-verifier
description: >-
  Prove all model traffic leaves through the sanctioned gateway, including
  tool-initiated and background calls, and that guardrail policies are
  attached and enforced. Use to attest gateway routing, to check whether a
  provider endpoint is directly reachable, or to evidence which guardrail
  policies are switched on.
allowed-tools: Bash, Read
```

# Llm Gateway Guardrail Verifier

**Controls:** Control 4 — gateway routing and guardrails

## Confidence: HIGH **only if** egress is enforced below the application

An application-layer gateway configuration is a routing preference. An agent
that can open a socket can bypass it by calling the provider directly. If the
allowlist is not enforced at the network layer, **downgrade this control to
PARTIAL** and say why.

## When to use this
When a deployment routes model traffic through a gateway and the attestation
wants to say so. Check first whether egress is enforced below the application:
if it is not, an agent opens a socket and the gateway is advisory, and this
control's confidence drops with it.

## Procedure

1. **Test reachability, do not read configuration.** From the deployment's
   network position, attempt to reach provider endpoints directly. Anything
   reachable that is not the gateway is a finding, regardless of what the
   configuration says.

2. **Cover the paths people forget.** Tool-initiated calls, background jobs,
   scheduled tasks, retry paths and sub-agents. A gateway that fronts the main
   request path and not the batch job is a gateway with a hole in it.

3. **Confirm guardrail attachment and version.** Record the guardrail ID and
   version actually attached to the route, not the one in the template.

4. **Enumerate enabled policies.** Content filters, prompt-attack filtering,
   denied topics, sensitive-information filters, contextual grounding. Record
   whether each is applied on **input, output, or both** — input-only filtering
   is a common and quiet gap.

5. **Confirm the action.** A filter set to observe rather than block is
   telemetry, not a control. Record the configured action per policy.

## Example

**Input** — the fixture committed at the top of [`scripts/llm_gateway_guardrail_verifier.py`](scripts/llm_gateway_guardrail_verifier.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
   the intended call           ALLOWED
   unregistered agent          denied at identity
   verb not permitted          denied at policy
   exfiltration destination    denied at egress
   over the per-target ceiling denied at budget

audit entries written: 1
credential held by the agent: never - attached at the gateway
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "deployment_id": "str",
  "gateway_enforced": true,
  "enforcement_layer": "network|application",
  "reachable_provider_findings": [{"endpoint": "str", "path": "direct|tool|background"}],
  "guardrail": {"id": "str", "version": "str",
                "policies": [{"name": "str", "applied_to": "input|output|both",
                              "action": "block|anonymize|observe"}]},
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Reading the route table instead of testing reachability.**
- **Missing the background path.** Scheduled and tool-initiated calls are the
  ones that bypass the gateway in practice.
- **Recording a guardrail as enforced when its action is observe.**

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/llm-gateway-guardrail-verifier/scripts/llm_gateway_guardrail_verifier.py
SCRIPT = "skills/attestation/llm-gateway-guardrail-verifier/scripts/llm_gateway_guardrail_verifier.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The skill loads and reports its shape. Its confidence is HIGH only where egress is enforced below the application — the gateway is a choke point because the network makes it one, not because the SDK was configured to point at it, and an application-level base URL is a default, not a control.

## Your turn

Count your agents. If it is more than five, work out how you would currently answer 'is egress control on for all of them' — and how long that would take.

---

**Next → [A3.8 · Shared infrastructure between agent runs](https://spbreed.github.io/cyber-commons/lessons/A3.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*